<a href="https://colab.research.google.com/github/ebolofis/Data-Science-Machine-Learning/blob/main/CAM_DS_C301_Implementing_bidirectional_RNNs_Demo_2_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<br>

**First things first** - please go to 'File' and select 'Save a copy in Drive' so that you have your own version of this activity set up and ready to use.
Remember to update the portfolio index link to your own work once completed!

# Demonstration 2.2.1 Implementing bidirectional RNNs

In this demonstration, we will delve into the practical aspects of implementing advanced recurrent neural network architectures using TensorFlow. We will specifically focus on bidirectional RNNs, exploring their necessity and advantages in various contexts.

By the end of this demonstration, you will gain a clear understanding of how to wrap a simple RNN, LSTM, or GRU within a bidirectional layer, and appreciate the trade-offs involved in terms of parameter count and computation time. This knowledge will equip you to make informed decisions when selecting the appropriate model architecture for your specific tasks.

The steps of the previous demonstrations have been left in for the purpose of your own comparison.


In [ ]:
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.

## Importing necessary libraries

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM,SimpleRNN, GRU, Bidirectional,SpatialDropout1D
from tensorflow.keras.datasets import imdb
import tensorflow as tf

In [ ]:
# Define a function to reset the session.
def reset_session():
    tf.keras.backend.clear_session()
    np.random.seed(42)
    tf.random.set_seed(42)

##  Loading the data set

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Create dataframes of the train and validation split.
text_train = dataset['train']['text']
label_train = dataset['train']['label']
text_test = dataset['test']['text']
label_test = dataset['test']['label']

In [ ]:
import pandas as pd
df_train = pd.DataFrame()
df_train['text'] = text_train
df_train['label'] = label_train

In [ ]:
df_train

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [ ]:
from sklearn.model_selection import train_test_split
train, valid = train_test_split(df_train,  test_size=0.2)

In [ ]:
text_train =  train['text'].tolist()
label_train = train['label'].tolist()
text_valid = valid['text'].tolist()
label_valid = valid['label'].tolist()


## Preprocessing the data
Sequences of numbers (words) are of different lengths. We need to pad them so that they have the same length for modelling.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Parameters.
vocab_size = 10000
max_length = 150
padding_type = 'post'
trunc_type = 'post'

# Initialise the tokeniser.
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(dataset['train']['text'])

# Tokenise the sentences and pad the sequences in the training set.
train_sequences = tokenizer.texts_to_sequences(text_train)
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)


# Tokenise the sentences and pad the sequences in the validation set.
valid_sequences = tokenizer.texts_to_sequences(text_valid)
valid_padded = pad_sequences(valid_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# Tokenise the sentences and pad the sequences in the test set.
test_sequences = tokenizer.texts_to_sequences(text_test )
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# Prepare the labels.
train_labels = np.array(label_train)
valid_labels = np.array(label_valid )
test_labels = np.array(label_test)


## Training the model with SimpleRNN

In [ ]:
reset_session()
embedding_vector_length = 32
model1 = Sequential()
model1.add(Embedding(vocab_size, embedding_vector_length, input_length=max_length))
model1.add(SimpleRNN(100))
model1.add(Dense(1, activation='sigmoid'))

model1.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model1.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 simple_rnn (SimpleRNN)      (None, 100)               13300     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 333401 (1.27 MB)
Trainable params: 333401 (1.27 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model1.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 26s 80ms/step - loss: 0.6977 - accuracy: 0.4981 - val_loss: 0.6947 - val_accuracy: 0.4936
Epoch 2/5
313/313 [==============================] - 23s 74ms/step - loss: 0.6931 - accuracy: 0.5222 - val_loss: 0.6930 - val_accuracy: 0.5044
Epoch 3/5
313/313 [==============================] - 22s 71ms/step - loss: 0.6885 - accuracy: 0.5455 - val_loss: 0.6933 - val_accuracy: 0.5106
Epoch 4/5
313/313 [==============================] - 23s 74ms/step - loss: 0.6750 - accuracy: 0.5751 - val_loss: 0.7102 - val_accuracy: 0.4994
Epoch 5/5
313/313 [==============================] - 22s 69ms/step - loss: 0.6431 - accuracy: 0.5975 - val_loss: 0.7241 - val_accuracy: 0.5212


In [ ]:
# Final evaluation of the model.
scores = model1.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 50.96%


## Training the model with LSTM

In [ ]:
reset_session()
# Create the model.
embedding_vector_length = 32
model2 = Sequential()
model2.add(Embedding(vocab_size, output_dim = embedding_vector_length, input_length=max_length))
model2.add(LSTM(100))
model2.add(Dense(1, activation='sigmoid'))
model2.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model2.summary())


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 lstm (LSTM)                 (None, 100)               53200     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 373301 (1.42 MB)
Trainable params: 373301 (1.42 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model2.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 84s 258ms/step - loss: 0.6771 - accuracy: 0.5687 - val_loss: 0.6639 - val_accuracy: 0.5864
Epoch 2/5
313/313 [==============================] - 85s 273ms/step - loss: 0.6481 - accuracy: 0.6208 - val_loss: 0.6852 - val_accuracy: 0.5408
Epoch 3/5
313/313 [==============================] - 80s 254ms/step - loss: 0.5369 - accuracy: 0.7462 - val_loss: 0.5008 - val_accuracy: 0.7866
Epoch 4/5
313/313 [==============================] - 80s 255ms/step - loss: 0.4801 - accuracy: 0.8001 - val_loss: 0.4823 - val_accuracy: 0.7970
Epoch 5/5
313/313 [==============================] - 79s 254ms/step - loss: 0.3956 - accuracy: 0.8486 - val_loss: 0.4597 - val_accuracy: 0.8046


In [ ]:
# Final evaluation of the model.
scores = model2.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 78.66%


## Training the model with GRU

In [ ]:
reset_session()
# Create the model.
embedding_vector_length = 32
model3 = Sequential()
model3.add(Embedding(vocab_size, output_dim = embedding_vector_length, input_length=max_length))
model3.add(GRU(100))
model3.add(Dense(1, activation='sigmoid'))
model3.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model3.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 gru (GRU)                   (None, 100)               40200     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 360301 (1.37 MB)
Trainable params: 360301 (1.37 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model3.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 78s 237ms/step - loss: 0.6658 - accuracy: 0.5648 - val_loss: 0.5341 - val_accuracy: 0.7582
Epoch 2/5
313/313 [==============================] - 73s 232ms/step - loss: 0.3949 - accuracy: 0.8368 - val_loss: 0.3810 - val_accuracy: 0.8426
Epoch 3/5
313/313 [==============================] - 70s 225ms/step - loss: 0.2758 - accuracy: 0.8960 - val_loss: 0.3504 - val_accuracy: 0.8532
Epoch 4/5
313/313 [==============================] - 70s 225ms/step - loss: 0.1991 - accuracy: 0.9309 - val_loss: 0.3516 - val_accuracy: 0.8560
Epoch 5/5
313/313 [==============================] - 71s 226ms/step - loss: 0.1417 - accuracy: 0.9542 - val_loss: 0.3849 - val_accuracy: 0.8492


In [ ]:
# Final evaluation of the model.
scores = model3.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 82.64%


## Bidirectional LSTM

In [ ]:
reset_session()
embedding_vector_length = 32
model4 = Sequential()
model4.add(Embedding(vocab_size, output_dim = embedding_vector_length, input_length=max_length))
model4.add(Bidirectional(LSTM(100)))  # Add a bidirectional LSTM.
model4.add(Dense(1, activation='sigmoid'))
model4.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model4.summary())


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 bidirectional (Bidirection  (None, 200)               106400    
 al)                                                             
                                                                 
 dense (Dense)               (None, 1)                 201       
                                                                 
Total params: 426601 (1.63 MB)
Trainable params: 426601 (1.63 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model4.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 130s 403ms/step - loss: 0.5136 - accuracy: 0.7349 - val_loss: 0.3991 - val_accuracy: 0.8360
Epoch 2/5
313/313 [==============================] - 126s 403ms/step - loss: 0.2985 - accuracy: 0.8853 - val_loss: 0.3366 - val_accuracy: 0.8510
Epoch 3/5
313/313 [==============================] - 129s 412ms/step - loss: 0.2274 - accuracy: 0.9157 - val_loss: 0.4487 - val_accuracy: 0.8376
Epoch 4/5
313/313 [==============================] - 127s 405ms/step - loss: 0.1750 - accuracy: 0.9366 - val_loss: 0.3990 - val_accuracy: 0.8426
Epoch 5/5
313/313 [==============================] - 129s 412ms/step - loss: 0.1406 - accuracy: 0.9500 - val_loss: 0.4387 - val_accuracy: 0.8438


In [ ]:
# Final evaluation of the model.
scores = model4.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 80.96%


## Bidirectional GRU

In [ ]:
reset_session()
embedding_vector_length = 32
model5 = Sequential()
model5.add(Embedding(vocab_size, output_dim = embedding_vector_length, input_length=max_length))
model5.add(Bidirectional(GRU(100)))  # Add a bidirectional GRU.
model5.add(Dense(1, activation='sigmoid'))
model5.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model5.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 bidirectional (Bidirection  (None, 200)               80400     
 al)                                                             
                                                                 
 dense (Dense)               (None, 1)                 201       
                                                                 
Total params: 400601 (1.53 MB)
Trainable params: 400601 (1.53 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model5.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 130s 404ms/step - loss: 0.5611 - accuracy: 0.6888 - val_loss: 0.3754 - val_accuracy: 0.8360
Epoch 2/5
313/313 [==============================] - 122s 389ms/step - loss: 0.3152 - accuracy: 0.8730 - val_loss: 0.3381 - val_accuracy: 0.8542
Epoch 3/5
313/313 [==============================] - 128s 410ms/step - loss: 0.2386 - accuracy: 0.9097 - val_loss: 0.3887 - val_accuracy: 0.8512
Epoch 4/5
313/313 [==============================] - 124s 396ms/step - loss: 0.1937 - accuracy: 0.9316 - val_loss: 0.4206 - val_accuracy: 0.8408
Epoch 5/5
313/313 [==============================] - 128s 408ms/step - loss: 0.1615 - accuracy: 0.9449 - val_loss: 0.4048 - val_accuracy: 0.8418


In [ ]:
# Final evaluation of the model.
scores = model5.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 81.40%


## Stacked LSTMs

In [ ]:
reset_session()
model6 = Sequential()
model6.add(Embedding(input_dim=vocab_size, output_dim=32, input_length=max_length))
model6.add(LSTM(100, return_sequences=True))
model6.add(LSTM(100))
model6.add(Dense(1, activation='sigmoid'))

model6.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model6.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 lstm (LSTM)                 (None, 150, 100)          53200     
                                                                 
 lstm_1 (LSTM)               (None, 100)               80400     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 453701 (1.73 MB)
Trainable params: 453701 (1.73 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model6.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 176s 546ms/step - loss: 0.5997 - accuracy: 0.6520 - val_loss: 0.4742 - val_accuracy: 0.7996
Epoch 2/5
313/313 [==============================] - 174s 556ms/step - loss: 0.3734 - accuracy: 0.8543 - val_loss: 0.3815 - val_accuracy: 0.8488
Epoch 3/5
313/313 [==============================] - 177s 566ms/step - loss: 0.2690 - accuracy: 0.9021 - val_loss: 0.3798 - val_accuracy: 0.8506
Epoch 4/5
313/313 [==============================] - 167s 535ms/step - loss: 0.2161 - accuracy: 0.9255 - val_loss: 0.3655 - val_accuracy: 0.8444
Epoch 5/5
313/313 [==============================] - 176s 564ms/step - loss: 0.1733 - accuracy: 0.9427 - val_loss: 0.4433 - val_accuracy: 0.8434


In [ ]:
# Final evaluation of the model.
scores = model6.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 81.42%


## Stacked GRUs

In [ ]:
reset_session()
model7 = Sequential()
model7.add(Embedding(input_dim=vocab_size, output_dim=32, input_length=max_length))
model7.add(GRU(100, return_sequences=True))
model7.add(GRU(100))
model7.add(Dense(1, activation='sigmoid'))

model7.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model7.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 gru (GRU)                   (None, 150, 100)          40200     
                                                                 
 gru_1 (GRU)                 (None, 100)               60600     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 420901 (1.61 MB)
Trainable params: 420901 (1.61 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model7.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 177s 555ms/step - loss: 0.6743 - accuracy: 0.5607 - val_loss: 0.6058 - val_accuracy: 0.6714
Epoch 2/5
313/313 [==============================] - 168s 539ms/step - loss: 0.4855 - accuracy: 0.7794 - val_loss: 0.6488 - val_accuracy: 0.6320
Epoch 3/5
313/313 [==============================] - 169s 538ms/step - loss: 0.3794 - accuracy: 0.8374 - val_loss: 0.3802 - val_accuracy: 0.8322
Epoch 4/5
313/313 [==============================] - 167s 534ms/step - loss: 0.2551 - accuracy: 0.9016 - val_loss: 0.3550 - val_accuracy: 0.8474
Epoch 5/5
313/313 [==============================] - 170s 544ms/step - loss: 0.1911 - accuracy: 0.9312 - val_loss: 0.3933 - val_accuracy: 0.8314


In [ ]:
# Final evaluation of the model.
scores = model7.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

## Stacked combination of LSTM and GRU

In [ ]:
reset_session()
model8 = Sequential()
model8.add(Embedding(input_dim=vocab_size, output_dim=32, input_length=max_length))
model8.add(LSTM(100, return_sequences=True))
model8.add(GRU(100, return_sequences = True))
model8.add(LSTM(100))
model8.add(Dense(1, activation='sigmoid'))

model8.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model8.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 spatial_dropout1d (Spatial  (None, 150, 32)           0         
 Dropout1D)                                                      
                                                                 
 lstm (LSTM)                 (None, 150, 100)          53200     
                                                                 
 gru (GRU)                   (None, 150, 100)          60600     
                                                                 
 lstm_1 (LSTM)               (None, 100)               80400     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                        

In [ ]:
model8.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 260s 811ms/step - loss: 0.6809 - accuracy: 0.5500 - val_loss: 0.6021 - val_accuracy: 0.7130
Epoch 2/5
313/313 [==============================] - 255s 815ms/step - loss: 0.4193 - accuracy: 0.8193 - val_loss: 0.3639 - val_accuracy: 0.8446
Epoch 3/5
313/313 [==============================] - 253s 806ms/step - loss: 0.3088 - accuracy: 0.8792 - val_loss: 0.3569 - val_accuracy: 0.8528
Epoch 4/5
313/313 [==============================] - 254s 811ms/step - loss: 0.2555 - accuracy: 0.9057 - val_loss: 0.3766 - val_accuracy: 0.8392
Epoch 5/5
313/313 [==============================] - 252s 806ms/step - loss: 0.2284 - accuracy: 0.9183 - val_loss: 0.3966 - val_accuracy: 0.8410


In [ ]:
# Final evaluation of the model.
scores = model8.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 81.27%


## Stacked bidirectional LSTM

In [ ]:
reset_session()
embedding_vector_length = 32
model9 = Sequential()
model9.add(Embedding(vocab_size, output_dim = embedding_vector_length, input_length=max_length))
model9.add(Bidirectional(LSTM(100, return_sequences = True)))  # Add a bidirectional LSTM.
model9.add(Bidirectional(LSTM(100)))  # Add a second bidirectional LSTM.
model9.add(Dense(1, activation='sigmoid'))
model9.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model9.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 bidirectional (Bidirection  (None, 150, 200)          106400    
 al)                                                             
                                                                 
 bidirectional_1 (Bidirecti  (None, 200)               240800    
 onal)                                                           
                                                                 
 dense (Dense)               (None, 1)                 201       
                                                                 
Total params: 667401 (2.55 MB)
Trainable params: 667401 (2.55 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model9.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 369s 1s/step - loss: 0.4847 - accuracy: 0.7556 - val_loss: 0.4109 - val_accuracy: 0.8276
Epoch 2/5
313/313 [==============================] - 359s 1s/step - loss: 0.3001 - accuracy: 0.8819 - val_loss: 0.3850 - val_accuracy: 0.8384
Epoch 3/5
313/313 [==============================] - 352s 1s/step - loss: 0.2298 - accuracy: 0.9146 - val_loss: 0.3652 - val_accuracy: 0.8524
Epoch 4/5
313/313 [==============================] - 369s 1s/step - loss: 0.1787 - accuracy: 0.9352 - val_loss: 0.4185 - val_accuracy: 0.8426
Epoch 5/5
313/313 [==============================] - 363s 1s/step - loss: 0.1422 - accuracy: 0.9496 - val_loss: 0.3849 - val_accuracy: 0.8416


In [ ]:
# Final evaluation of the model.
scores = model9.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 81.07%


## Key information
This demonstration illustrated how to implement Bidirectional RNN's (including GRU's and LSTM's)

## Reflect
What are the practical applications of these techniques?
> Select the pen from the toolbar to add your entry.